<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

Система учёта фактур (Invoice Management System)

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>


Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.
    
Требования к базовому классу Invoice:

Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

Методы:

CalculateTotal(): метод для расчета общей суммы по фактуре.

AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.


Требования к производным классам:
1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные
атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен
быть переопределен для добавления информации о дате поставки товара
при добавлении позиции.
2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные
атрибуты, такие как Дата оказания услуги (ServiceDate).
Метод RemoveLine() должен быть переопределен для добавления
информации о причине аннулирования услуги при удалении позиции.
3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс):
Должна содержать дополнительные атрибуты, такие как Наличие возврата
(ReturnAllowed). Метод CalculateTotal() должен быть переопределен для
учета возможного возврата товара или услуги при расчете общей суммы.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [15]:
using System;
using System.Collections.Generic;

public class LineItem
{
    public string Name { get; set; }
    public decimal Price { get; set; }
    public int Quantity { get; set; }

    public LineItem(string name, decimal price, int quantity)
    {
        Name = name;
        Price = price;
        Quantity = quantity;
    }

    public decimal GetAmount() => Price * Quantity;

    public override string ToString() => $"{Name} — {Quantity} x {Price:0.00} = {GetAmount():0.00}";
}

public class Invoice
{
    public string InvoiceNumber { get; set; }
    public DateTime IssueDate { get; set; }
    public decimal TotalAmount { get; protected set; }

    protected List<LineItem> Lines { get; } = new List<LineItem>();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    public virtual decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (var line in Lines)
        {
            total += line.GetAmount();
        }
        TotalAmount = total;
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        Lines.Add(lineItem);
        Console.WriteLine($"Добавлена позиция: {lineItem}");
        CalculateTotal();
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        Lines.Remove(lineItem);
        Console.WriteLine($"Удалена позиция: {lineItem}");
        CalculateTotal();
    }

    public void PrintInfo()
    {
        Console.WriteLine($"Фактура №{InvoiceNumber} от {IssueDate:d}, сумма: {TotalAmount:0.00}");
    }
}

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        Lines.Add(lineItem);
        Console.WriteLine($"Добавлена позиция товара: {lineItem} (дата поставки: {SupplyDate:d})");
        CalculateTotal();
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    public void RemoveLine(LineItem lineItem, string cancellationReason)
    {
        Lines.Remove(lineItem);
        Console.WriteLine($"Услуга аннулирована: {lineItem}. Причина: {cancellationReason}");
        CalculateTotal();
    }

    public override void RemoveLine(LineItem lineItem)
    {
        RemoveLine(lineItem, "причина не указана");
    }
}

public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public decimal ReturnAmount { get; set; }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed)
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();

        if (ReturnAllowed && ReturnAmount > 0)
        {
            total -= ReturnAmount;
            Console.WriteLine($"Учтён возврат на сумму {ReturnAmount:0.00}");
        }

        TotalAmount = total;
        return TotalAmount;
    }
}


Console.WriteLine("=== Товарная фактура ===");
var goodsInvoice = new GoodsInvoice("G-001", DateTime.Now, DateTime.Now.AddDays(3));
goodsInvoice.AddLine(new LineItem("Ноутбук", 55000m, 1));
goodsInvoice.AddLine(new LineItem("Мышь", 1200m, 2));
goodsInvoice.PrintInfo();

Console.WriteLine();
Console.WriteLine("=== Услуговая фактура ===");
var serviceInvoice = new ServiceInvoice("S-001", DateTime.Now, DateTime.Now);
var consulting = new LineItem("Консультация", 3000m, 1);
serviceInvoice.AddLine(consulting);
serviceInvoice.AddLine(new LineItem("Настройка ПО", 5000m, 1));
serviceInvoice.RemoveLine(consulting, "Клиент отменил встречу");
serviceInvoice.PrintInfo();

Console.WriteLine();
Console.WriteLine("=== Комбинированная фактура ===");
var combinedInvoice = new CombinedInvoice("C-001", DateTime.Now, true);
combinedInvoice.AddLine(new LineItem("Принтер", 12000m, 1));
combinedInvoice.AddLine(new LineItem("Обслуживание принтера", 2000m, 1));
combinedInvoice.ReturnAmount = 2000m;
combinedInvoice.CalculateTotal();
combinedInvoice.PrintInfo();

Console.WriteLine();
Console.WriteLine("=== Полиморфизм: работа с разными фактурами через базовый тип Invoice ===");
List<Invoice> invoices = new List<Invoice> { goodsInvoice, serviceInvoice, combinedInvoice };
foreach (var inv in invoices)
{
    inv.PrintInfo();
}

=== Товарная фактура ===
Добавлена позиция товара: Ноутбук — 1 x 55000,00 = 55000,00 (дата поставки: 13.09.2026)
Добавлена позиция товара: Мышь — 2 x 1200,00 = 2400,00 (дата поставки: 13.09.2026)
Фактура №G-001 от 10.09.2026, сумма: 57400,00

=== Услуговая фактура ===
Добавлена позиция: Консультация — 1 x 3000,00 = 3000,00
Добавлена позиция: Настройка ПО — 1 x 5000,00 = 5000,00
Услуга аннулирована: Консультация — 1 x 3000,00 = 3000,00. Причина: Клиент отменил встречу
Фактура №S-001 от 10.09.2026, сумма: 5000,00

=== Комбинированная фактура ===
Добавлена позиция: Принтер — 1 x 12000,00 = 12000,00
Добавлена позиция: Обслуживание принтера — 1 x 2000,00 = 2000,00
Учтён возврат на сумму 2000,00
Фактура №C-001 от 10.09.2026, сумма: 12000,00

=== Полиморфизм: работа с разными фактурами через базовый тип Invoice ===
Фактура №G-001 от 10.09.2026, сумма: 57400,00
Фактура №S-001 от 10.09.2026, сумма: 5000,00
Фактура №C-001 от 10.09.2026, сумма: 12000,00
